# 4장 — BPE 최적화와 병렬 토크나이징

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/zerokaraLLM/blob/main/notebooks/ch04_tokenizer_optimization.ipynb)

이 노트북은 『밑바닥부터 시작하는 딥러닝 6』의 공식 코드 저장소를 기준으로 구성했습니다. T4에서 실행하기 어렵다는 이유로 알고리즘이나 모델 구조를 토이 버전으로 바꾸지 않습니다.

- 기준 upstream commit: `c9b6e2ed531b08dd9f451a091a34e9645148e2e2`
- 포함한 장 코드 파일 수: **6개**
- 함께 펼쳐서 보여주는 공통 모듈 수: **1개**


## 노트북 구성 원칙

1. 공식 `.py`의 모델 구조와 계산 로직을 그대로 유지합니다.
2. 함수·클래스·실행부를 셀 단위로 나눠 위에서 아래로 읽기 쉽게 배치합니다.
3. 일본어 자연어 주석은 한국어로 바꾸며, 변수명·수식·텐서 shape 같은 기술 표기는 유지합니다.
4. 공통 `codebot` / `storybot` 모듈도 외부 파일 뒤에 숨기지 않고 이 노트북에서 직접 확인할 수 있게 합니다.
5. T4에서 시간이 오래 걸리는 전체 학습 스케줄도 기본값 자체를 임의 축소하지 않습니다.


## 0. Colab 환경 준비

먼저 공식 저장소를 고정된 커밋으로 준비하고 현재 런타임의 GPU를 확인합니다.


In [ ]:
from pathlib import Path
import os
import subprocess

UPSTREAM_COMMIT = 'c9b6e2ed531b08dd9f451a091a34e9645148e2e2'
WORKDIR = Path('/content/deep-learning-from-scratch-6')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/oreilly-japan/deep-learning-from-scratch-6.git', str(WORKDIR)], check=True)
    subprocess.run(['git', '-C', str(WORKDIR), 'checkout', '--quiet', UPSTREAM_COMMIT], check=True)

os.chdir(WORKDIR)
print('작업 경로:', Path.cwd())

try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA 사용 가능:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch 확인 중 오류:', exc)


## 1. 이 장에서 사용하는 공통 구현

장 코드가 import하는 로컬 모듈을 먼저 읽습니다. 긴 파일도 클래스·함수 단위로 나눠 표시합니다.


### `storybot/tokenizer.py`

이 파일은 이 장에서 사용하는 공통 구현입니다. 파일 전체를 숨기지 않고 구성 요소별로 나누어 확인합니다.


#### 필요한 라이브러리와 모듈 불러오기


In [ ]:
%%writefile storybot/tokenizer.py
import os
import pickle
from multiprocessing import Pool
import shutil
from collections import defaultdict
import regex as re
from tqdm import tqdm
import numpy as np


#### `pretokenize()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    for m in re.finditer(pattern, text):
        yield m.group(0)


#### `count_pairs()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


#### `merge()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


#### `find_chunk_boundaries()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def find_chunk_boundaries(file_path, num_chunks, end_token="<|endoftext|>"):
    byte_end_token = end_token.encode("utf-8")

    with open(file_path, "rb") as file:  # 이 코드 단계의 동작을 확인하는 예시
        # 이 코드 단계의 동작을 확인하는 예시
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // num_chunks

        # 이 코드 단계의 동작을 확인하는 예시
        chunk_boundaries = [i * chunk_size for i in range(num_chunks)]
        chunk_boundaries.append(file_size)  # 이 코드 단계의 동작을 확인하는 예시

        buffer_size = 4096  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for bi in range(1, len(chunk_boundaries) - 1):
            chunk_position = chunk_boundaries[bi]
            file.seek(chunk_position)  # 이 코드 단계의 동작을 확인하는 예시

            while True:
                buffer = file.read(buffer_size)  # 이 코드 단계의 동작을 확인하는 예시

                # 이 코드 단계의 동작을 확인하는 예시
                if buffer == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                end_position = buffer.find(byte_end_token)
                if end_position != -1:
                    # 이 코드 단계의 동작을 확인하는 예시
                    chunk_boundaries[bi] = chunk_position + end_position
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                chunk_position += buffer_size

    # 이 코드 단계의 동작을 확인하는 예시
    return sorted(set(chunk_boundaries))


#### `process_single_chunk()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def process_single_chunk(file_path, start, end, end_token):
    """1つのチャンクを処理する関数"""
    pretoken_counts = defaultdict(int)

    # 이 코드 단계의 동작을 확인하는 예시
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 이 코드 단계의 동작을 확인하는 예시
        texts = chunk_text.split(end_token)

        # 이 코드 단계의 동작을 확인하는 예시
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts


#### `pretoken_chunk()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def pretoken_chunk(args):
    file_path, start, end, end_token = args
    pretoken_counts = defaultdict(int)

    # 이 코드 단계의 동작을 확인하는 예시
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 이 코드 단계의 동작을 확인하는 예시
        texts = chunk_text.split(end_token)

        # 이 코드 단계의 동작을 확인하는 예시
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts


#### `train_bpe()` 함수 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

def train_bpe(file_path, vocab_size, end_token="<|endoftext|>", num_processes=8, num_chunks=8):
    # 이 코드 단계의 동작을 확인하는 예시
    chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
    total_chunks = len(chunk_boundaries) - 1

    chunk_info_list = []
    for i in range(total_chunks):
        start = chunk_boundaries[i]
        end = chunk_boundaries[i + 1]
        chunk_info_list.append((file_path, start, end, end_token))

    # 이 코드 단계의 동작을 확인하는 예시
    with Pool(processes=num_processes) as pool:
        all_results = list(tqdm(pool.imap(pretoken_chunk, chunk_info_list), total=len(chunk_info_list), desc="Pretokenizing"))

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    for chunk_result in all_results:
        for pretoken, count in chunk_result.items():
            pretoken_counts[pretoken] += count

    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}


    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # 캐시

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # 이 코드 단계의 동작을 확인하는 예시
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # 이 코드 단계의 동작을 확인하는 예시
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 이 코드 단계의 동작을 확인하는 예시
            ids_counts[tuple(new_ids)] = ids_count  # 이 코드 단계의 동작을 확인하는 예시

            # 이 코드 단계의 동작을 확인하는 예시
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 이 코드 단계의 동작을 확인하는 예시
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules


#### `BPETokenizer` 클래스 구현


In [ ]:
%%writefile -a storybot/tokenizer.py

class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 이 코드 단계의 동작을 확인하는 예시

        while len(ids) > 1:
            # 이 코드 단계의 동작을 확인하는 예시
            counts = count_pairs(ids)

            # 이 코드 단계의 동작을 확인하는 예시
            best_pair = min(counts, key=get_merge_priority)

            # 이 코드 단계의 동작을 확인하는 예시
            if best_pair not in self.merge_rules:
                break

            # 이 코드 단계의 동작을 확인하는 예시
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def _encode_chunk(self, args):
        """チャンクを処理してディスクにキャッシュ"""
        file_path, start, end, cache_dir, chunk_idx = args

        with open(file_path, "rb") as f:
            f.seek(start)
            chunk_byte = f.read(end - start)
            chunk_text = chunk_byte.decode("utf-8", errors="ignore")

            # 이 코드 단계의 동작을 확인하는 예시
            ids = self.encode(chunk_text)

        # 이 코드 단계의 동작을 확인하는 예시
        cache_file = os.path.join(cache_dir, f"chunk_{chunk_idx:05d}.npy")
        np.array(ids, dtype=np.uint16).tofile(cache_file)

        return cache_file, len(ids)


    def encode_file(self, file_path, output_file,
                                    num_processes=4, num_chunks=64,
                                   cache_dir="bpe_cache"):

        # 이 코드 단계의 동작을 확인하는 예시
        os.makedirs(cache_dir, exist_ok=True)

        try:
            # 이 코드 단계의 동작을 확인하는 예시
            chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
            total_chunks = len(chunk_boundaries) - 1

            chunk_info_list = []
            for i in range(total_chunks):
                start = chunk_boundaries[i]
                end = chunk_boundaries[i + 1]
                chunk_info_list.append((file_path, start, end, cache_dir, i))

            with Pool(processes=num_processes) as pool:
                cache_results = list(tqdm(
                    pool.imap(self._encode_chunk, chunk_info_list),
                    total=len(chunk_info_list),
                    desc="Encoding chunks"
                ))

            # 이 코드 단계의 동작을 확인하는 예시
            cache_files = [r[0] for r in cache_results]
            token_counts = [r[1] for r in cache_results]
            total_tokens = sum(token_counts)

            # 이 코드 단계의 동작을 확인하는 예시
            dtype = np.uint16
            arr = np.memmap(output_file, dtype=dtype, mode='w+', shape=(total_tokens,))

            # 이 코드 단계의 동작을 확인하는 예시
            # 이 코드 단계의 동작을 확인하는 예시
            idx = 0
            for cache_file in cache_files:
                chunk_data = np.fromfile(cache_file, dtype=dtype)
                arr[idx : idx + len(chunk_data)] = chunk_data
                idx += len(chunk_data)

            arr.flush()
            del arr

        finally:
            # 이 코드 단계의 동작을 확인하는 예시
            shutil.rmtree(cache_dir)

        return total_tokens

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


## 2. 장별 실습 코드

공식 저장소의 장 코드를 파일 순서대로 모두 다룹니다.


## `ch04/01_bpe_optimize.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from collections import defaultdict
import regex as re
from tqdm import tqdm


### `pretokenize()` 함수 구현


In [ ]:


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 이 코드 단계의 동작을 확인하는 예시
    texts = input_text.split(end_token)

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    for text in tqdm(texts, desc="Pretokenizing"):  # 이 코드 단계의 동작을 확인하는 예시
        for pretoken in pretokenize(text):
            pretoken_counts[pretoken] += 1

    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}

    num_merges = vocab_size - 256 - 1
    merge_rules = {}

    for step in tqdm(range(num_merges), desc="Training BPE"):
        # 이 코드 단계의 동작을 확인하는 예시
        pair_counts = defaultdict(int)
        for ids, count in ids_counts.items():
            count_pairs(ids, count, pair_counts)

        # 이 코드 단계의 동작을 확인하는 예시
        if not pair_counts:
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))

        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        new_ids_counts = defaultdict(int)
        for ids, count in ids_counts.items():
            new_ids = merge(ids, best_pair, new_id)  # 이 코드 단계의 동작을 확인하는 예시
            new_ids_counts[tuple(new_ids)] += count
        ids_counts = new_ids_counts

    return merge_rules


### 설정 및 값 준비: `vocab_size`


In [ ]:


vocab_size = 1000  # 어휘 크기의설정


### 설정 및 값 준비: `file_path`


In [ ]:
file_path = "codebot/tiny_codes.txt"


### 설정 및 값 준비: `text`


In [ ]:
text = open(file_path).read()


### 설정 및 값 준비: `merge_rules`


In [ ]:
merge_rules = train_bpe(text, vocab_size)


## `ch04/02_bpe_cache.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from collections import defaultdict
import regex as re
from tqdm import tqdm


### `pretokenize()` 함수 구현


In [ ]:


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    return re.findall(pattern, text)


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(input_text, vocab_size, end_token="<|endoftext|>"):
    # 이 코드 단계의 동작을 확인하는 예시
    texts = input_text.split(end_token)

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    for text in tqdm(texts, desc="Pretokenizing"):  # 이 코드 단계의 동작을 확인하는 예시
        for pretoken in pretokenize(text):
            pretoken_counts[pretoken] += 1

    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}

    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # 캐시

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # 이 코드 단계의 동작을 확인하는 예시
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # 이 코드 단계의 동작을 확인하는 예시
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 이 코드 단계의 동작을 확인하는 예시
            ids_counts[tuple(new_ids)] = ids_count  # 이 코드 단계의 동작을 확인하는 예시

            # 이 코드 단계의 동작을 확인하는 예시
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 이 코드 단계의 동작을 확인하는 예시
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules


### 설정 및 값 준비: `vocab_size`


In [ ]:


vocab_size = 1000  # 어휘 크기의설정


### 설정 및 값 준비: `file_path`


In [ ]:
file_path = "codebot/tiny_codes.txt"


### 설정 및 값 준비: `text`


In [ ]:
text = open(file_path).read()


### 설정 및 값 준비: `merge_rules`


In [ ]:
merge_rules = train_bpe(text, vocab_size)


### 마지막 실행 코드


In [ ]:

# 참고: vocab_size = 10000
# file_path = "storybot/tiny_stories_train.txt"
# 참고: text = open(file_path).read()
# 참고: merge_rules = train_bpe(text, vocab_size)


## `ch04/03_bpe_chunk.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import os
from collections import defaultdict
import regex as re
from tqdm import tqdm


### `pretokenize()` 함수 구현


In [ ]:


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    for m in re.finditer(pattern, text):
        yield m.group(0)


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `find_chunk_boundaries()` 함수 구현


In [ ]:

def find_chunk_boundaries(file_path, num_chunks, end_token="<|endoftext|>"):
    byte_end_token = end_token.encode("utf-8")

    with open(file_path, "rb") as file:  # 이 코드 단계의 동작을 확인하는 예시
        # 이 코드 단계의 동작을 확인하는 예시
        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // num_chunks

        # 이 코드 단계의 동작을 확인하는 예시
        chunk_boundaries = [i * chunk_size for i in range(num_chunks)]
        chunk_boundaries.append(file_size)  # 이 코드 단계의 동작을 확인하는 예시

        buffer_size = 4096  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for bi in range(1, len(chunk_boundaries) - 1):
            chunk_position = chunk_boundaries[bi]
            file.seek(chunk_position)  # 이 코드 단계의 동작을 확인하는 예시

            while True:
                buffer = file.read(buffer_size)  # 이 코드 단계의 동작을 확인하는 예시

                # 이 코드 단계의 동작을 확인하는 예시
                if buffer == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                end_position = buffer.find(byte_end_token)
                if end_position != -1:
                    # 이 코드 단계의 동작을 확인하는 예시
                    chunk_boundaries[bi] = chunk_position + end_position
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                chunk_position += buffer_size

    # 이 코드 단계의 동작을 확인하는 예시
    return sorted(set(chunk_boundaries))


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(file_path, vocab_size, end_token="<|endoftext|>"):
    chunk_boundaries = find_chunk_boundaries(file_path, num_chunks=64)

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    with open(file_path, "rb") as f:
        total_chunks = len(chunk_boundaries) - 1

        for i in tqdm(range(total_chunks), desc="Pretokenizing"):
            start = chunk_boundaries[i]
            end = chunk_boundaries[i+1]

            f.seek(start)
            chunk_byte = f.read(end - start)
            chunk_text = chunk_byte.decode("utf-8", errors="ignore")

            # 이 코드 단계의 동작을 확인하는 예시
            texts = chunk_text.split(end_token)
            # 이 코드 단계의 동작을 확인하는 예시
            for text in texts:
                for pretoken in pretokenize(text):
                    pretoken_counts[pretoken] += 1


    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}

    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # 캐시

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # 이 코드 단계의 동작을 확인하는 예시
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # 이 코드 단계의 동작을 확인하는 예시
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 이 코드 단계의 동작을 확인하는 예시
            ids_counts[tuple(new_ids)] = ids_count  # 이 코드 단계의 동작을 확인하는 예시

            # 이 코드 단계의 동작을 확인하는 예시
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 이 코드 단계의 동작을 확인하는 예시
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules


### 설정 및 값 준비: `file_path`


In [ ]:


file_path = "storybot/tiny_stories_train.txt"


### 설정 및 값 준비: `vocab_size`


In [ ]:
vocab_size = 10000  # 어휘 크기의설정


### 설정 및 값 준비: `merge_rules`


In [ ]:
merge_rules = train_bpe(file_path, vocab_size)


## `ch04/04_bpe_parallel.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

import os
from multiprocessing import Pool
from collections import defaultdict
import regex as re
from tqdm import tqdm
import pickle


### `pretokenize()` 함수 구현


In [ ]:


def pretokenize(text):
    pattern = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""
    for m in re.finditer(pattern, text):
        yield m.group(0)


### `count_pairs()` 함수 구현


In [ ]:

def count_pairs(ids, weight=1, counts=None):
    if counts is None:
        counts = defaultdict(int)

    for pair in zip(ids, ids[1:]):
        counts[pair] += weight
    return counts


### `merge()` 함수 구현


In [ ]:

def merge(ids, pair, new_id):
    merged_ids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and (ids[i], ids[i+1]) == pair:
            merged_ids.append(new_id)
            i += 2
        else:
            merged_ids.append(ids[i])
            i += 1
    return merged_ids


### `find_chunk_boundaries()` 함수 구현


In [ ]:

def find_chunk_boundaries(file_path, num_chunks, end_token="<|endoftext|>"):
    byte_end_token = end_token.encode("utf-8")

    with open(file_path, "rb") as file:

        file.seek(0, os.SEEK_END)
        file_size = file.tell()
        file.seek(0)

        chunk_size = file_size // num_chunks


        chunk_boundaries = [i * chunk_size for i in range(num_chunks)]
        chunk_boundaries.append(file_size)  # 이 코드 단계의 동작을 확인하는 예시

        buffer_size = 4096  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for bi in range(1, len(chunk_boundaries) - 1):
            chunk_position = chunk_boundaries[bi]
            file.seek(chunk_position)  # 이 코드 단계의 동작을 확인하는 예시

            while True:
                buffer = file.read(buffer_size)  # 이 코드 단계의 동작을 확인하는 예시

                # 이 코드 단계의 동작을 확인하는 예시
                if buffer == b"":
                    chunk_boundaries[bi] = file_size
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                end_position = buffer.find(byte_end_token)
                if end_position != -1:
                    # 이 코드 단계의 동작을 확인하는 예시
                    chunk_boundaries[bi] = chunk_position + end_position
                    break

                # 이 코드 단계의 동작을 확인하는 예시
                chunk_position += buffer_size

    # 이 코드 단계의 동작을 확인하는 예시
    return sorted(set(chunk_boundaries))


### `pretoken_chunk()` 함수 구현


In [ ]:

def pretoken_chunk(args):
    file_path, start, end, end_token = args
    pretoken_counts = defaultdict(int)

    # 이 코드 단계의 동작을 확인하는 예시
    with open(file_path, "rb") as f:
        f.seek(start)
        chunk_byte = f.read(end - start)
        chunk_text = chunk_byte.decode("utf-8", errors="ignore")

        # 이 코드 단계의 동작을 확인하는 예시
        texts = chunk_text.split(end_token)

        # 이 코드 단계의 동작을 확인하는 예시
        for text in texts:
            for pretoken in pretokenize(text):
                pretoken_counts[pretoken] += 1

    return pretoken_counts


### `train_bpe()` 함수 구현


In [ ]:

def train_bpe(file_path, vocab_size, end_token="<|endoftext|>", num_processes=8, num_chunks=64):
    # 이 코드 단계의 동작을 확인하는 예시
    chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
    total_chunks = len(chunk_boundaries) - 1

    chunk_info_list = []
    for i in range(total_chunks):
        start = chunk_boundaries[i]
        end = chunk_boundaries[i + 1]
        chunk_info_list.append((file_path, start, end, end_token))

    # 이 코드 단계의 동작을 확인하는 예시
    with Pool(processes=num_processes) as pool:
        all_results = list(tqdm(pool.imap(pretoken_chunk, chunk_info_list), total=len(chunk_info_list), desc="Pretokenizing"))

    # 이 코드 단계의 동작을 확인하는 예시
    pretoken_counts = defaultdict(int)
    for chunk_result in all_results:
        for pretoken, count in chunk_result.items():
            pretoken_counts[pretoken] += count

    # 이 코드 단계의 동작을 확인하는 예시
    ids_counts = {tuple(pretoken.encode("utf-8")): count for pretoken, count in pretoken_counts.items()}


    num_merges = vocab_size - 256 - 1
    merge_rules = {}
    pair_to_ids = defaultdict(set)  # 캐시

    pair_counts = defaultdict(int)
    for ids, count in ids_counts.items():
        count_pairs(ids, count, pair_counts)
        for pair in zip(ids, ids[1:]):  # 이 코드 단계의 동작을 확인하는 예시
            pair_to_ids[pair].add(ids)

    for step in tqdm(range(num_merges), desc="Training BPE"):
        if not pair_counts:  # 이 코드 단계의 동작을 확인하는 예시
            break

        # 이 코드 단계의 동작을 확인하는 예시
        # 참고: best_pair = max(pair_counts, key=pair_counts.get)
        best_pair = max(pair_counts, key=lambda pair: (pair_counts[pair], pair[0], pair[1]))
        new_id = 256 + step
        merge_rules[best_pair] = new_id

        # 이 코드 단계의 동작을 확인하는 예시
        affected_ids = pair_to_ids[best_pair]
        del pair_to_ids[best_pair]  # 이 코드 단계의 동작을 확인하는 예시

        # 이 코드 단계의 동작을 확인하는 예시
        for ids in affected_ids:
            ids_count = ids_counts[tuple(ids)]
            new_ids = merge(ids, best_pair, new_id)

            del ids_counts[tuple(ids)]  # 이 코드 단계의 동작을 확인하는 예시
            ids_counts[tuple(new_ids)] = ids_count  # 이 코드 단계의 동작을 확인하는 예시

            # 이 코드 단계의 동작을 확인하는 예시
            old_counts = count_pairs(ids)
            for pair, count in old_counts.items():
                pair_counts[pair] -= count * ids_count
                if pair_counts[pair] <= 0:
                    del pair_counts[pair]
                pair_to_ids[pair].discard(tuple(ids))

            # 이 코드 단계의 동작을 확인하는 예시
            new_counts = count_pairs(new_ids)
            for pair, count in new_counts.items():
                pair_counts[pair] += count * ids_count
                pair_to_ids[pair].add(tuple(new_ids))

    return merge_rules


### 조건에 따른 실행


In [ ]:


if __name__ == '__main__':
    vocab_size = 10000
    file_path = "storybot/tiny_stories_train.txt"
    # 이 코드 단계의 동작을 확인하는 예시
    merge_rules = train_bpe(file_path, vocab_size, num_processes=8)

    with open("storybot/merge_rules.pkl", "wb") as f:
        pickle.dump(merge_rules, f)


## `ch04/06_encode_optimize.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from storybot.tokenizer import pretokenize, count_pairs, merge

import pickle
import regex as re
from tqdm import tqdm


### `BPETokenizer` 클래스 구현


In [ ]:


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 이 코드 단계의 동작을 확인하는 예시

        while len(ids) > 1:
            # 이 코드 단계의 동작을 확인하는 예시
            counts = count_pairs(ids)

            # 이 코드 단계의 동작을 확인하는 예시
            best_pair = min(counts, key=get_merge_priority)

            # 이 코드 단계의 동작을 확인하는 예시
            if best_pair not in self.merge_rules:
                break

            # 이 코드 단계의 동작을 확인하는 예시
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### 조건에 따른 실행


In [ ]:


if __name__ == "__main__":
    tokenizer = BPETokenizer.load_from("codebot/merge_rules.pkl")

    file_path = "codebot/tiny_codes.txt"
    text = open(file_path).read()
    ids = tokenizer.encode(text, show_progress=True)
    print(len(ids))


## `ch04/07_encode_parallel.py`

원본 스크립트를 노트북 흐름에 맞춰 구성 요소별 셀로 나눴습니다. 실행 코드 자체는 주석을 제외하고 변경하지 않았습니다.


### 필요한 라이브러리와 모듈 불러오기


In [ ]:
import os, sys


### 실행 및 결과 확인


In [ ]:
os.chdir(os.path.join(os.path.dirname(os.path.abspath(__file__)), '..'))


### 실행 및 결과 확인


In [ ]:
sys.path.append('.')


### 필요한 라이브러리와 모듈 불러오기


In [ ]:

from storybot.tokenizer import pretokenize, count_pairs, merge, find_chunk_boundaries

import os
import pickle
from multiprocessing import Pool
import shutil
import regex as re
from tqdm import tqdm
import numpy as np


### `BPETokenizer` 클래스 구현


In [ ]:


class BPETokenizer:
    def __init__(self, merge_rules, end_token="<|endoftext|>"):
        self.merge_rules = merge_rules
        self.end_token = end_token
        self.end_token_id = 256 + len(merge_rules)

        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        for (id1, id2), new_id in merge_rules.items():
            self.id_to_bytes[new_id] = self.id_to_bytes[id1] + self.id_to_bytes[id2]
        self.id_to_bytes[self.end_token_id] = self.end_token.encode("utf-8")

        self.vocab_size = len(self.id_to_bytes)

    @staticmethod
    def load_from(filepath):
        with open(filepath, "rb") as f:
            merge_rules = pickle.load(f)
        return BPETokenizer(merge_rules)

    def _encode_text(self, text):
        ids = list(text.encode("utf-8"))

        def get_merge_priority(pair):
            return self.merge_rules.get(pair, float('inf'))  # 이 코드 단계의 동작을 확인하는 예시

        while len(ids) > 1:
            # 이 코드 단계의 동작을 확인하는 예시
            counts = count_pairs(ids)

            # 이 코드 단계의 동작을 확인하는 예시
            best_pair = min(counts, key=get_merge_priority)

            # 이 코드 단계의 동작을 확인하는 예시
            if best_pair not in self.merge_rules:
                break

            # 이 코드 단계의 동작을 확인하는 예시
            new_id = self.merge_rules[best_pair]
            ids = merge(ids, best_pair, new_id)

        return ids

    def _encode_chunk(self, args):
        """チャンクを処理してディスクにキャッシュ"""
        file_path, start, end, cache_dir, chunk_idx = args

        with open(file_path, "rb") as f:
            f.seek(start)
            chunk_byte = f.read(end - start)
            chunk_text = chunk_byte.decode("utf-8", errors="ignore")

            # 이 코드 단계의 동작을 확인하는 예시
            ids = self.encode(chunk_text)

        # 이 코드 단계의 동작을 확인하는 예시
        cache_file = os.path.join(cache_dir, f"chunk_{chunk_idx:05d}.npy")
        np.array(ids, dtype=np.uint16).tofile(cache_file)

        return cache_file, len(ids)


    def encode_file(self, file_path, output_file,
                                    num_processes=8, num_chunks=64,
                                   cache_dir="bpe_cache"):

        # 이 코드 단계의 동작을 확인하는 예시
        os.makedirs(cache_dir, exist_ok=True)

        try:
            # 이 코드 단계의 동작을 확인하는 예시
            chunk_boundaries = find_chunk_boundaries(file_path, num_chunks)
            total_chunks = len(chunk_boundaries) - 1

            chunk_info_list = []
            for i in range(total_chunks):
                start = chunk_boundaries[i]
                end = chunk_boundaries[i + 1]
                chunk_info_list.append((file_path, start, end, cache_dir, i))

            with Pool(processes=num_processes) as pool:
                cache_results = list(tqdm(
                    pool.imap(self._encode_chunk, chunk_info_list),
                    total=len(chunk_info_list),
                    desc="Encoding chunks"
                ))

            # 이 코드 단계의 동작을 확인하는 예시
            cache_files = [r[0] for r in cache_results]
            token_counts = [r[1] for r in cache_results]
            total_tokens = sum(token_counts)

            # memmap파일생성
            dtype = np.uint16
            arr = np.memmap(output_file, dtype=dtype, mode='w+', shape=(total_tokens,))

            # 이 코드 단계의 동작을 확인하는 예시
            idx = 0
            for cache_file in cache_files:
                chunk_data = np.fromfile(cache_file, dtype=dtype)
                arr[idx : idx + len(chunk_data)] = chunk_data
                idx += len(chunk_data)

            arr.flush()
            del arr

        finally:
            # 이 코드 단계의 동작을 확인하는 예시
            shutil.rmtree(cache_dir)

        return total_tokens

    def encode(self, input_text, show_progress=False):
        pattern = '(' + re.escape(self.end_token) + ')'
        texts = re.split(pattern, input_text)
        all_ids = []

        # 이 코드 단계의 동작을 확인하는 예시
        texts = tqdm(texts) if show_progress else texts

        for text in texts:
            if text == self.end_token:
                all_ids.append(self.end_token_id)
            else:
                # 이 코드 단계의 동작을 확인하는 예시
                for pretoken in pretokenize(text):
                    ids = self._encode_text(pretoken)
                    all_ids.extend(ids)

        return all_ids

    def decode(self, ids):
        byte_list = [self.id_to_bytes[i] for i in ids]
        text_bytes = b"".join(byte_list)
        text = text_bytes.decode("utf-8", errors="replace")
        return text


### 조건에 따른 실행


In [ ]:


if __name__ == '__main__':
    tokenizer = BPETokenizer.load_from("storybot/merge_rules.pkl")

    tokenizer.encode_file(
        "storybot/tiny_stories_train.txt",
        "storybot/tiny_stories_train.bin", num_processes=8)

    tokenizer.encode_file(
        "storybot/tiny_stories_valid.txt",
        "storybot/tiny_stories_valid.bin", num_processes=8)


## T4 실행 메모

위 코드는 공식 구현의 모델 구조·알고리즘·기본 하이퍼파라미터를 보존합니다. 학습 시간이 긴 셀은 T4에서도 실행 자체는 가능할 수 있지만 전체 스텝 완주에는 시간이 많이 필요할 수 있습니다. 이 노트북은 빠른 실행을 위해 모델을 임의로 축소하거나 핵심 계산을 생략하지 않습니다.
